In [1]:
!pip install -q google-generativeai PyPDF2 gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.6 MB/s eta 0:00:00


In [2]:
import google.generativeai as genai
import os
import pandas as pd
import numpy as np
from PyPDF2 import PdfReader

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
from google.colab import userdata
api_key = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=api_key)
print("google.generativeai library imported and configured.")

google.generativeai library imported and configured.


In [4]:
import time

def call_with_retry(fn, *args, retries=2, delay=3, **kwargs):
    """Call fn(*args, **kwargs); retry on transient Gemini errors (503/429/timeout).
    Re-raises the last error if all attempts fail, so callers still see a real failure
    instead of this silently returning nothing."""
    last_err = None
    for attempt in range(retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            transient = any(s in str(e) for s in ("503", "429", "Timeout", "Deadline", "overloaded", "UNAVAILABLE"))
            if not transient or attempt == retries:
                raise
            print(f"Transient error ({type(e).__name__}), retrying in {delay}s... (attempt {attempt + 1}/{retries})")
            time.sleep(delay)
    raise last_err

In [ ]:
# Single place to change the generation model - swap here if the current one is hitting rate limits/overload
# (e.g. "gemini-3.5-flash" if a heavier model is needed).
MODEL_NAME = "gemini-3.5-flash-lite"

## Етап 1: зчитування даних

In [5]:
df = pd.DataFrame(columns=['Title','Text'])
df

,Title,Text


In [6]:
# Brightdesk demo docs, loaded inline (no PDF upload needed for this demo run).
brightdesk_docs = {
    'Onboarding Guide': "# Brightdesk — New Hire Onboarding Guide\n\n## Before Your First Day\n- IT will email your account credentials 3 business days before your start date. If you haven't received them by then, contact it-support@brightdesk.example.\n- Bring a government-issued ID for identity verification on day one.\n- Your manager will send a calendar invite for a 30-minute welcome call on your first morning.\n\n## First Week Checklist\n1. Complete the HR intake form (personal details, emergency contact, tax forms).\n2. Set up your company laptop and enroll in two-factor authentication.\n3. Meet with your onboarding buddy (assigned by your manager) for a team walkthrough.\n4. Review the Employee Handbook and acknowledge receipt in the HR portal.\n5. Schedule 1:1s with your direct manager and skip-level manager within the first two weeks.\n\n## Probation Period\nAll new hires have a 90-day probation period. Formal check-ins happen at day 30, day 60, and day 90. Full benefits eligibility (health insurance, 401k/pension matching) begins on day 1, but probation performance is reviewed before the role is confirmed as permanent.\n\n## Equipment\nStandard equipment (laptop, monitor, headset) is provided by IT. Requests for additional equipment go through the #it-requests channel and are typically fulfilled within 5 business days.\n",
    'Leave and Vacation Policy': "# Brightdesk — Leave & Vacation Policy\n\n## Paid Time Off (PTO)\nFull-time employees accrue 24 calendar days of paid annual leave per year (the statutory minimum under Ukrainian labor law), credited at 2 days per month worked. Unused PTO up to 5 days may be carried over into the next calendar year; anything beyond that is forfeited unless local law requires otherwise.\n\n## Requesting Leave\nSubmit leave requests through the HR portal at least 5 business days in advance for planned leave. Requests are approved by your direct manager. Leave taken without prior approval (except sick leave) may be recorded as unpaid.\n\n## Sick Leave\nEmployees receive 10 paid sick days per year, separate from PTO. A doctor's note is required for absences longer than 3 consecutive days. Sick leave does not need to be scheduled in advance.\n\n## Parental Leave\n- Primary caregiver: 16 weeks paid leave.\n- Secondary caregiver: 6 weeks paid leave.\nParental leave can be taken within 12 months of the birth or adoption date and does not need to be used consecutively.\n\n## Public Holidays\nBrightdesk observes 11 public holidays per year, published annually in the HR portal calendar. Employees working in a different country follow their local public holiday calendar.\n\n## Bereavement Leave\nUp to 5 paid days for the loss of an immediate family member, and 2 paid days for extended family, at manager's discretion for edge cases.\n",
    'Health Insurance and Benefits': '# Brightdesk — Health Insurance & Benefits\n\n## Medical Insurance\nBrightdesk covers 100% of the employee\'s medical insurance premium and 50% of dependents\' premiums under the standard plan. Coverage begins on the first day of employment, with no waiting period. Enrollment or plan changes are handled through the benefits portal within 30 days of a qualifying life event (marriage, birth, relocation).\n\n## Dental & Vision\nOptional dental and vision add-ons are available at the employee\'s cost, with a 20% company subsidy. These can be added during open enrollment (every November) or within 30 days of hire.\n\n## Retirement / Pension Matching\nBrightdesk matches retirement contributions up to 4% of base salary, vested immediately (no vesting schedule). Contributions can be adjusted at any time through the payroll portal.\n\n## Wellness Stipend\nEmployees receive a $50/month wellness stipend reimbursable for gym memberships, fitness classes, or mental health app subscriptions. Submit receipts through the expense portal under category "Wellness."\n\n## Learning & Development Budget\nEach employee has an annual $1,000 learning budget for courses, certifications, or conference tickets, subject to manager approval. Unused budget does not roll over to the next year.\n',
    'Remote and Hybrid Work Policy': "# Brightdesk — Remote & Hybrid Work Policy\n\n## Work Location\nEmployees on hybrid contracts are expected in the office 2 days per week (Tuesday and Thursday, unless otherwise agreed with a manager). Fully remote roles are defined at the offer stage and do not have an office attendance requirement.\n\n## Working Hours\nCore collaboration hours are 11:00–16:00 in the employee's local timezone. Outside core hours, employees may structure their schedule flexibly as long as deliverables and meeting commitments are met.\n\n## Equipment for Remote Work\nRemote employees receive a one-time $300 home office stipend for desk/chair setup, plus standard IT-issued equipment (laptop, monitor). Requests go through the #it-requests channel.\n\n## Working From Abroad\nEmployees may work from another country for up to 30 consecutive days per year without prior HR approval, provided it does not affect tax residency. Longer periods require sign-off from HR and Legal due to tax and compliance implications.\n\n## Internet & Connectivity\nA $40/month home internet stipend is available for fully remote employees, reimbursed through the expense portal.\n",
}

for title, text in brightdesk_docs.items():
    text = text.replace('\n', ' ')
    new_row = pd.DataFrame({'Title': [title], 'Text': [text]})
    df = pd.concat([df, new_row], ignore_index=True)


In [7]:
df

,Title,Text
0,Onboarding Guide,# Brightdesk — New Hire Onboarding Guide ## B...
1,Leave and Vacation Policy,# Brightdesk — Leave & Vacation Policy ## Pai...
2,Health Insurance and Benefits,# Brightdesk — Health Insurance & Benefits ##...
3,Remote and Hybrid Work Policy,# Brightdesk — Remote & Hybrid Work Policy ##...


In [8]:
print(df['Text'].iloc[1])

# Brightdesk — Leave & Vacation Policy  ## Paid Time Off (PTO) Full-time employees accrue 24 calendar days of paid annual leave per year (the statutory minimum under Ukrainian labor law), credited at 2 days per month worked. Unused PTO up to 5 days may be carried over into the next calendar year; anything beyond that is forfeited unless local law requires otherwise.  ## Requesting Leave Submit leave requests through the HR portal at least 5 business days in advance for planned leave. Requests are approved by your direct manager. Leave taken without prior approval (except sick leave) may be recorded as unpaid.  ## Sick Leave Employees receive 10 paid sick days per year, separate from PTO. A doctor's note is required for absences longer than 3 consecutive days. Sick leave does not need to be scheduled in advance.  ## Parental Leave - Primary caregiver: 16 weeks paid leave. - Secondary caregiver: 6 weeks paid leave. Parental leave can be taken within 12 months of the birth or adoption dat

## Етап 2: Векторизація

In [9]:
texts = df['Text']
task_type = 'RETRIEVAL_DOCUMENT'
result = genai.embed_content(model="gemini-embedding-001", content=texts)
print(result)


{'embedding': [[-0.006957309, 0.015669828, -0.002305022, -0.05786128, 0.023529692, 0.021365006, -0.0048412452, -0.010775757, -0.037184883, -0.006749296, -0.05041439, -0.010716306, 0.00018510461, -0.007115277, 0.11918452, -0.016847137, 0.014053316, 0.0037684322, 0.0012318222, -0.019463938, -0.0091748675, 0.014780136, 0.004766676, -0.0023347707, -0.022118734, -0.030557282, 0.035106838, 0.030003343, 0.042310067, 0.0007536542, 0.0010637754, 0.02587698, 0.0078955535, 0.0282977, -0.0031126952, 0.023079716, -0.010645984, 0.0008762546, -0.013065552, 0.016391497, 0.008287095, 0.014236358, -0.017534593, -0.020157427, -0.008688459, -0.009051389, -0.006668721, 0.0042255097, -0.0076902406, 0.021984987, -0.01772308, 0.01311187, -0.007437147, -0.19306765, -0.0048237303, -0.027916854, 0.006469652, -0.0028410193, 0.01916372, -0.028317709, 0.0011445108, 0.027103094, -0.046611074, -0.028726097, -0.009938895, -0.026199052, 0.0010534151, -0.005843079, 0.02205556, -0.019225411, 0.004598728, 0.0069517074, -0

In [10]:
print(result['embedding'])

[[-0.006957309, 0.015669828, -0.002305022, -0.05786128, 0.023529692, 0.021365006, -0.0048412452, -0.010775757, -0.037184883, -0.006749296, -0.05041439, -0.010716306, 0.00018510461, -0.007115277, 0.11918452, -0.016847137, 0.014053316, 0.0037684322, 0.0012318222, -0.019463938, -0.0091748675, 0.014780136, 0.004766676, -0.0023347707, -0.022118734, -0.030557282, 0.035106838, 0.030003343, 0.042310067, 0.0007536542, 0.0010637754, 0.02587698, 0.0078955535, 0.0282977, -0.0031126952, 0.023079716, -0.010645984, 0.0008762546, -0.013065552, 0.016391497, 0.008287095, 0.014236358, -0.017534593, -0.020157427, -0.008688459, -0.009051389, -0.006668721, 0.0042255097, -0.0076902406, 0.021984987, -0.01772308, 0.01311187, -0.007437147, -0.19306765, -0.0048237303, -0.027916854, 0.006469652, -0.0028410193, 0.01916372, -0.028317709, 0.0011445108, 0.027103094, -0.046611074, -0.028726097, -0.009938895, -0.026199052, 0.0010534151, -0.005843079, 0.02205556, -0.019225411, 0.004598728, 0.0069517074, -0.0026115673, -

In [11]:
def embed_text(text: str) -> np.ndarray:
    resp = call_with_retry(
        genai.embed_content,
        model="gemini-embedding-001",
        content=text,
        task_type=task_type,   # "retrieval_query" or "retrieval_document"
    )
    return np.asarray(resp["embedding"], dtype=np.float32)

In [12]:
df['Embeddings'] = df['Text'].apply(embed_text)
df

,Title,Text,Embeddings
0,Onboarding Guide,# Brightdesk — New Hire Onboarding Guide ## B...,"[-0.0075476617, 0.016903909, -0.001111857, -0...."
1,Leave and Vacation Policy,# Brightdesk — Leave & Vacation Policy ## Pai...,"[0.004197855, 0.026593978, 0.0062915646, -0.05..."
2,Health Insurance and Benefits,# Brightdesk — Health Insurance & Benefits ##...,"[-0.022602076, 0.024628248, -0.003922993, -0.0..."
3,Remote and Hybrid Work Policy,# Brightdesk — Remote & Hybrid Work Policy ##...,"[0.00039880569, 0.016733184, 0.005057843, -0.0..."


In [13]:
def query_similarity_score(query,vector):
    # query --> 'How can I take a sick leave?' TEXT STR
    # vector --> our existing Array (list)

    # Query STR --> LIST
    query_embedding = embed_text(query)

    #SIMILARITY
    return np.dot(query_embedding,vector)

In [14]:
query = "Скільки в мене на рік доступно вихідних?"
df['Similarity'] = df['Embeddings'].apply(lambda vector: query_similarity_score(query,vector))

In [15]:
df

,Title,Text,Embeddings,Similarity
0,Onboarding Guide,# Brightdesk — New Hire Onboarding Guide ## B...,"[-0.0075476617, 0.016903909, -0.001111857, -0....",0.686474
1,Leave and Vacation Policy,# Brightdesk — Leave & Vacation Policy ## Pai...,"[0.004197855, 0.026593978, 0.0062915646, -0.05...",0.766189
2,Health Insurance and Benefits,# Brightdesk — Health Insurance & Benefits ##...,"[-0.022602076, 0.024628248, -0.003922993, -0.0...",0.682433
3,Remote and Hybrid Work Policy,# Brightdesk — Remote & Hybrid Work Policy ##...,"[0.00039880569, 0.016733184, 0.005057843, -0.0...",0.706529


In [16]:
#now let`s extract `the text we need to have as context to answer our question
df.sort_values('Similarity',ascending=False)
df.sort_values('Similarity',ascending=False)[['Title','Text']]
df.sort_values('Similarity',ascending=False)[['Title','Text']].iloc[0]

,1
Title,Leave and Vacation Policy
Text,# Brightdesk — Leave & Vacation Policy ## Pai...


In [17]:
def  most_similar_doc(query):
    df['Similarity'] = df['Embeddings'].apply(lambda vector: query_similarity_score(query,vector))
    title = df.sort_values('Similarity',ascending=False)[['Title','Text']].iloc[0]['Title']
    text = df.sort_values('Similarity',ascending=False)[['Title','Text']].iloc[0]['Text']
    return title,text

In [18]:
title, text = most_similar_doc('Хочу відвідати конференцію. Чи може компанія мені це покрити?')
print(title,text)

Leave and Vacation Policy # Brightdesk — Leave & Vacation Policy  ## Paid Time Off (PTO) Full-time employees accrue 24 calendar days of paid annual leave per year (the statutory minimum under Ukrainian labor law), credited at 2 days per month worked. Unused PTO up to 5 days may be carried over into the next calendar year; anything beyond that is forfeited unless local law requires otherwise.  ## Requesting Leave Submit leave requests through the HR portal at least 5 business days in advance for planned leave. Requests are approved by your direct manager. Leave taken without prior approval (except sick leave) may be recorded as unpaid.  ## Sick Leave Employees receive 10 paid sick days per year, separate from PTO. A doctor's note is required for absences longer than 3 consecutive days. Sick leave does not need to be scheduled in advance.  ## Parental Leave - Primary caregiver: 16 weeks paid leave. - Secondary caregiver: 6 weeks paid leave. Parental leave can be taken within 12 months of

## Етап 3: Генерація відповіді

In [23]:
def rag_implement(query):
    #Retrival Document
    title,text = most_similar_doc(query)
    #AG - Model
    model = genai.GenerativeModel(MODEL_NAME)
    #Augment --> prompt engineering
    prompt = f'Answer this query:\n{query}. \n Only use this context to answer: \n{text}'
    #Sourcing(title) - what doc was used
    response = call_with_retry(model.generate_content, prompt)
    # return f'{response.text}\n\n Source Doc Title: {title}'
    return response.text,title

In [29]:
print(rag_implement('А чи є курси німецької?'))

('На основі наданого контексту інформація про курси німецької мови відсутня.', 'Leave and Vacation Policy')


## Chunking

Get the most relevant chunks to the query

In [ ]:
def get_relevant_chunks(query: str, num_chunks: int = 3):
    q = np.asarray(embed_text(query), dtype=np.float32)   # one API call
    sims = EMB_MATRIX @ q                                 # dot product vs all chunks
    top_idx = np.argsort(-sims)[:num_chunks]
    top = df_chunks.iloc[top_idx]
    return top["Text"].tolist(), top["Title"].tolist()

## Improving chunking - by split per sentence + adding preprocessing

In [ ]:
import nltk
import re
from nltk.tokenize import sent_tokenize

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
def text_splitter(text, chunk_size=500, overlap=50):
    chunks = []
    # Ensure text is a string
    if not isinstance(text, str):
        text = str(text)

    # Preprocessing: remove extra whitespace and unnecessary symbols
    # Replace multiple spaces/newlines with a single space.
    text = re.sub(r'[\s]+', ' ', text)
    # Keep alphanumeric characters, spaces, and common punctuation.
    # Added Cyrillic character range (а-яА-Я) for Ukrainian text.
    text = re.sub(r'[^a-zA-Zа-яА-ЯІіЇїЄє0-9\s.,!?;:]', '', text)
    text = text.strip()

    sentences = sent_tokenize(text)

    current_sentence_index = 0
    while current_sentence_index < len(sentences):
        current_chunk_sentences = []
        current_chunk_length = 0

        # Add sentences from the end of the previous chunk for overlap
        if len(chunks) > 0:
            last_chunk_sentences = sent_tokenize(chunks[-1])
            temp_overlap_chars = 0
            overlap_sentences_for_this_chunk = []

            # Iterate backwards in the previous chunk's sentences to find enough for overlap
            for i in range(len(last_chunk_sentences) - 1, -1, -1):
                sentence = last_chunk_sentences[i]
                # Add 1 for the space that will be between sentences when joined
                if temp_overlap_chars + len(sentence) + (1 if temp_overlap_chars > 0 else 0) <= overlap:
                    overlap_sentences_for_this_chunk.insert(0, sentence) # Add to front to maintain order
                    temp_overlap_chars += len(sentence) + (1 if temp_overlap_chars > 0 else 0)
                else:
                    break
            current_chunk_sentences.extend(overlap_sentences_for_this_chunk)
            current_chunk_length = len(" ".join(current_chunk_sentences)) if current_chunk_sentences else 0

        # Now add new sentences from the `sentences` list until chunk_size is met
        temp_sentence_idx = current_sentence_index
        while temp_sentence_idx < len(sentences):
            sentence = sentences[temp_sentence_idx]
            # Check if adding this sentence will exceed chunk_size
            # Add 1 for the space that will be between sentences when joined
            if current_chunk_length + len(sentence) + (1 if current_chunk_sentences else 0) <= chunk_size:
                current_chunk_sentences.append(sentence)
                current_chunk_length += len(sentence) + (1 if current_chunk_sentences else 0)
                temp_sentence_idx += 1
            else:
                break # Current chunk is full enough

        # Handle cases where a single sentence is larger than chunk_size, or if no sentences were added
        if not current_chunk_sentences and current_sentence_index < len(sentences):
            # If the current_chunk_sentences is empty, it means the very first sentence is too long or no sentences were processed yet
            # Take the current sentence, even if it's too long, as a single chunk
            current_chunk_sentences.append(sentences[current_sentence_index])
            temp_sentence_idx = current_sentence_index + 1 # Advance pointer

        if current_chunk_sentences:
            chunks.append(" ".join(current_chunk_sentences))

        current_sentence_index = temp_sentence_idx # Move to the point where the next chunk should start

    return chunks

In [ ]:
chunked_data = []

for index, row in df.iterrows():
    title = row['Title']
    full_text = row['Text']

    # Split the full text into chunks
    chunks = text_splitter(full_text)

    # For each chunk, create a new entry with title, chunk text, and embedding
    for chunk in chunks:
        chunk_embedding = embed_text(chunk)
        chunked_data.append({'Title': title, 'Text': chunk, 'Embeddings': chunk_embedding})

# Create a new DataFrame from the chunked data
df_chunks = pd.DataFrame(chunked_data)

display(df_chunks.head())

In [ ]:
df_chunks = df_chunks.reset_index(drop=True)

df_chunks["Embeddings"] = df_chunks["Embeddings"].apply(lambda x: np.asarray(x, dtype=np.float32))

EMB_MATRIX = np.vstack(df_chunks["Embeddings"].to_numpy())
EMB_MATRIX_N = EMB_MATRIX / (np.linalg.norm(EMB_MATRIX, axis=1, keepdims=True) + 1e-12)

In [ ]:
get_relevant_chunks("Скільки в мене на рік доступно вихідних?")

In [28]:
def rag_implement_chunked(query):
    # Retrieval of relevant chunks
    relevant_texts, source_titles = get_relevant_chunks(query, num_chunks=3)

    # Concatenate relevant texts for the prompt
    context = " ".join(relevant_texts)

    # AG - Model
    model = genai.GenerativeModel(MODEL_NAME)

    # Augment --> prompt engineering
    prompt = f'Answer this query:\n{query}. \n Only use this context to answer: \n{context}'

    response = call_with_retry(model.generate_content, prompt)

    # Sourcing (titles of documents from which chunks were taken)
    source_titles = df_chunks[df_chunks['Text'].isin(relevant_texts)]['Title'].unique().tolist()

    return response.text, "\n".join(sorted(set(source_titles)))

In [ ]:
rag_implement_chunked("Скільки в мене на рік доступно вихідних?")

## Extending the user query

In [27]:
def extend_query(original_query):
    model = genai.GenerativeModel(MODEL_NAME)
    prompt = f"""Given the query: '{original_query}', generate 2-3 alternative ways to phrase this question that could be used to find similar information.
    Return each variant on a new line and include the original query as one of the options.
    Do not include any commentary, introductory or concluding text, just the list of queries.
    The result must be in the language of the original query, no translations. If it`s in English, return variants in English, if in Ukrainian, return in Ukrainian, etc."""
    response = call_with_retry(model.generate_content, prompt)
    extended_queries = [q.strip() for q in response.text.split('\n') if q.strip()]
    if original_query not in extended_queries:
        extended_queries.insert(0, original_query) # Ensure original query is always included
    return extended_queries

In [ ]:
extend_query("Чи є курси німецької?")

In [26]:
def rag_implement_chunked_extended(query: str):
    extended_queries = extend_query(query)

    combined_context_chunks = []
    source_titles = []

    # get 1 top chunk per extended query
    for ext_query in extended_queries:
        texts, titles = get_relevant_chunks(ext_query, num_chunks=1)  # unpack
        combined_context_chunks.extend(texts)
        source_titles.extend(titles)

    # remove duplicate chunks while preserving order
    seen = set()
    unique_chunks = []
    for t in combined_context_chunks:
        if t not in seen:
            unique_chunks.append(t)
            seen.add(t)

    final_relevant_texts = unique_chunks[:3]
    context = " ".join(final_relevant_texts)

    model = genai.GenerativeModel(MODEL_NAME)
    prompt = f"Answer this query:\n{query}\nOnly use this context to answer:\n{context}"
    response = call_with_retry(model.generate_content, prompt)

    used_titles = df_chunks[df_chunks["Text"].isin(final_relevant_texts)]["Title"].unique().tolist()

    return response.text, "\n".join(sorted(set(used_titles)))

In [ ]:
rag_implement_chunked_extended("Чи є курси німецької?")

## Demo queries — quick text proof before launching the chat UI
Run these to get printed Q&A with cited source, useful for a screenshot or for pasting straight into the portfolio write-up.

In [ ]:
question_en = "How many vacation days do I get per year, and can I carry them over?"
answer, sources = rag_implement_chunked_extended(question_en)
print(f"Q: {question_en}\nA: {answer}\nSource(s): {sources}")


In [ ]:
question_ua = "Скільки днів мені потрібно бути в офісі щотижня при гібридному графіку?"
answer, sources = rag_implement_chunked_extended(question_ua)
print(f"Q: {question_ua}\nA: {answer}\nSource(s): {sources}")


In [ ]:
import gradio as gr

description = "Welcome to HR policies bureau. Here you can ask anything about onboarding process, medical insuranse, leave policies and benefits."
title = "HR Wiki 👾"
output1 = gr.Textbox(label="Answer")
output2 = gr.Textbox(label="Where it comes from?")
gr.Interface(fn=rag_implement_chunked,
             title=title,
             description=description,
             inputs=["text"],
             outputs=[output1,output2]).launch(debug=True)

## Slack demo — one-off bot for a portfolio screenshot

Temporary: runs only while this cell is executing, in this Colab session. Not a deployed bot — for capturing a real Q&A screenshot in Slack, then stop the cell (Runtime → Interrupt execution) and revoke the Slack app token afterwards.

**Before running, in Slack (api.slack.com/apps):**
1. Create New App → From scratch → pick your workspace
2. Settings → Socket Mode → turn on → generate an App-Level Token with scope `connections:write` (starts with `xapp-`)
3. OAuth & Permissions → Bot Token Scopes → add `chat:write`, `channels:history`, `im:history`, `im:read`, `im:write`
4. Event Subscriptions → Subscribe to bot events → add `message.channels` and `message.im`
5. Install App to Workspace → copy the Bot User OAuth Token (starts with `xoxb-`)
6. Create (or pick) one dedicated test channel and invite the bot: `/invite @YourBotName`
7. Get that channel's ID: right-click the channel name → View channel details → the ID is at the bottom of that panel (starts with `C`)

**In Colab Secrets (🔑), add three new secrets** (same way as `GOOGLE_API_KEY`, Notebook access ON):
- `SLACK_BOT_TOKEN` = the `xoxb-...` token
- `SLACK_APP_TOKEN` = the `xapp-...` token
- `SLACK_TEST_CHANNEL_ID` = the channel ID from step 7 (e.g. `C0123456789`)

The bot only answers inside that one channel — see the channel check in the code below. That way, if this cell is accidentally left running, it doesn't start answering HR questions in every channel or DM it happens to be invited to.

In [ ]:
!pip install slack_bolt slack_sdk -q

from slack_bolt import App
from slack_bolt.adapter.socket_mode import SocketModeHandler
from google.colab import userdata

slack_bot_token = userdata.get('SLACK_BOT_TOKEN')
slack_app_token = userdata.get('SLACK_APP_TOKEN')
allowed_channel_id = userdata.get('SLACK_TEST_CHANNEL_ID')

if not allowed_channel_id:
    raise ValueError(
        "SLACK_TEST_CHANNEL_ID secret is not set. Add it (see markdown cell above) "
        "before starting the bot — this keeps it from answering in every channel it's invited to."
    )

slack_app = App(token=slack_bot_token)

@slack_app.event("message")
def handle_message(event, say):
    # ignore the bot's own messages and message edits/subtypes
    if event.get("bot_id") or event.get("subtype"):
        return

    # only answer in the one dedicated test channel, ignore every other
    # channel/DM the bot might be invited to (accidentally or otherwise)
    if event.get("channel") != allowed_channel_id:
        return

    question = event.get("text", "").strip()
    if not question:
        return

    try:
        answer, sources = rag_implement_chunked_extended(question)
        say(f"*Answer:* {answer}\n*Source:* {sources}")
    except Exception as e:
        # don't leave the person hanging on a transient Gemini error (503/timeout/etc.)
        say("Sorry, I hit an error answering that (the model may be temporarily "
            f"overloaded). Try again in a moment. ({type(e).__name__})")

print(f"Bot is listening in channel {allowed_channel_id} only. Go ask it a question there.")
print("Stop this cell (Runtime -> Interrupt execution) once you have your screenshot.")

handler = SocketModeHandler(slack_app, slack_app_token)
handler.start()

In [31]:
print(rag_implement('Скільки днів відпустки на рік?'))

('Згідно з контекстом, повні дні оплачуваної щорічної відпустки (Paid Time Off, PTO) становлять **24 календарні дні на рік**.', 'Leave and Vacation Policy')
